# Explore calculated results with Plotly

Run **Restart Kernel and Run All** to create an analysis and explore its
report. This notebook is self-contained: the model declarations and all nine
mock rows are copied from [poc_extented_model_test.py](poc_extented_model_test.py),
with formatting changes only. The experiment itself is not imported or modified.
The filename in this repository contains `extented`.

The example compares **August and September 2026** for **Operations**:

- Permanent row cost: rate × worked hours + bonus.
- Temporary (`gpd`) row cost: regular hours × rate + overtime hours × rate.
- Permanent composition: grade, then `reg_coeff` within grade, then employee
  membership within the cell. Here `reg_coeff` is a **categorical dimension**,
  not a multiplier in the payroll formula or an identified geographic region.

You will see the one-call report, reusable waterfalls, driver contributions
by team, copied tables, figure styling, offline HTML and a bundle round trip.
Only the analysis cell runs attribution; subsequent reporting reads its results.

## 1. Use the notebook environment

From `production/`, install `python -m pip install -e '.[notebook]'`, then
select that environment's kernel. For hosted GitLab installation, use the
[deployment guide](../docs/gitlab-deployment.md) and
[reporting requirements](../runtime/requirements-reporting.txt).
Restart the kernel after installation.

`plotly_mimetype` suits JupyterLab and VS Code. In classic Notebook use
`notebook` for embedded JavaScript. If your hosted frontend cannot display
Plotly, the HTML example below works without a CDN. These settings do not
change Plotly's global renderer.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from IPython.display import display

from labor_cost import (
    AnalysisRequest,
    DatasetSnapshot,
    Factor,
    FieldSpec,
    Game,
    ModelGraph,
    ModelSpec,
    Population,
    RowCost,
    SourceSchema,
    Switch,
    aligned_mean,
    analyze,
    child,
    compile_model,
    export_run,
    field,
    load_run,
    local_game,
    organization_game,
    outer_factors,
    plot_driver_by_team,
    plot_organization_waterfall,
    plot_team_waterfall,
    report,
    select,
    sum_over,
    value,
)

renderer = "plotly_mimetype"
currency_label = "Currency"
selected_teams = ["Operations"]

## 2. Declare the same model

These setup cells preserve the experiment's fields, populations, factors,
switches, hierarchy and completion policy. Expand them to inspect the code.
The population and source category are `gpd`; its game ID is `temp_game`.
Labels such as “Temporary workforce mix” are presentation text, not IDs.
For model-authoring details see [GUIDE.MD](GUIDE.MD).

In [2]:
source = SourceSchema(
    dimensions={
        "grade": "grade",
        "reg_coeff": "reg_coeff",
        },
)

populations = (
    Population(
        "permanent",
        {"employment_type": "permanent"},
    ),
    Population(
        "gpd",
        {"employment_type": "gpd"},
    ),
)

fields = (
    FieldSpec(
        "permanent_rate", "stavka_oklad",
        population="permanent",
        units="currency/time", constraint="nonnegative",
    ),
    FieldSpec(
        "permanent_hours", "vyrabotka_act",
        population="permanent",
        units="time", constraint="nonnegative",
    ),
    FieldSpec(
        "permanent_bonus", "bonus_kpi",
        population="permanent",
        units="currency", constraint="signed",
    ),
    FieldSpec(
        "temporary_hours", "temporary_hours",
        population="gpd",
        units="time", constraint="nonnegative",
    ),
    FieldSpec(
        "temporary_rate", "temporary_rate",
        population="gpd",
        units="currency/time", constraint="positive",
    ),
    FieldSpec(
        "overtime_hours", "overtime_hours",
        population="gpd",
        units="time", constraint="nonnegative",
    ),
    FieldSpec(
        "overtime_rate", "overtime_rate",
        population="gpd",
        units="currency/time", constraint="positive",
    ),
)

row_costs = (
    RowCost(
        "permanent",
        field("permanent_rate") * field("permanent_hours")
        + field("permanent_bonus"),
    ),
    RowCost(
        "gpd",
        field("temporary_hours") * field("temporary_rate")
        + field("overtime_hours") * field("overtime_rate"),
    ),
)

In [3]:
factors = outer_factors() + (
    # Permanent/temporary shares within each team.
    Factor(
        "employment_mix",
        "population_mix",
        dimensions=("employment_kind",),
        members=("permanent", "gpd"),
    ),

    # Grade shares within each team's permanent population.
    Factor(
        "grade_mix",
        "distribution",
        population="permanent",
        dimensions=("grade",),
    ),

    Factor(
        "region_within_grade_mix",
        "distribution",
        population="permanent",
        dimensions=("reg_coeff",),
        given=("grade",),
        fallbacks=(()),
    ),

    # Which permanent employees contribute within each cell.
    Factor(
        "permanent_members",
        "membership",
        population="permanent",
        dimensions=("grade", "reg_coeff"),
        fallbacks=((),),
    ),

    # Preserve permanent rate/hour pairing by employee.
    Factor(
        "permanent_rate",
        "aligned",
        population="permanent",
        source="permanent_rate",
    ),
    Factor(
        "permanent_hours",
        "aligned",
        population="permanent",
        source="permanent_hours",
    ),
    Factor(
        "permanent_bonus",
        "aligned",
        population="permanent",
        source="permanent_bonus",
    ),

    # Temporary workers have no categorical breakdown.
    Factor(
        "temporary_members",
        "membership",
        population="gpd",
    ),

    # Preserve all four temporary inputs by employee.
    Factor(
        "temporary_hours",
        "aligned",
        population="gpd",
        source="temporary_hours",
    ),
    Factor(
        "temporary_rate",
        "aligned",
        population="gpd",
        source="temporary_rate",
    ),
    Factor(
        "overtime_hours",
        "aligned",
        population="gpd",
        source="overtime_hours",
    ),
    Factor(
        "overtime_rate",
        "aligned",
        population="gpd",
        source="overtime_rate",
    ),
)

In [4]:
permanent_unit_cost = sum_over(
    value("grade_mix")
    * value("region_within_grade_mix")
    * (
        aligned_mean(
            value("permanent_rate") * value("permanent_hours")
            + value("permanent_bonus"),
            "permanent_members",
        )
    ),
    "grade",
    "reg_coeff",
)

temporary_unit_cost = aligned_mean(
    value("temporary_hours") * value("temporary_rate")
    + value("overtime_hours") * value("overtime_rate"),
    "temporary_members",
)

permanent_game = Game(
    name="permanent",
    population="permanent",
    formula=permanent_unit_cost,
    switches=(
        Switch(
            "grade_mix",
            label="Permanent grade composition",
        ),
        Switch(
            "region_within_grade_mix",
            label="Permanent region composition within grade",
        ),
        Switch(
            "permanent_members",
            label="Workforce mix within cell",
        ),
        Switch(
            "permanent_rate",
            label="Permanent hourly rates",
        ),
        Switch(
            "permanent_hours",
            label="Permanent worked hours",
        ),
        Switch(
            "permanent_bonus",
            label="Permanent bonuses",
        ),
    ),
)

temporary_game = Game(
    name="temp_game",
    population="gpd",
    formula=temporary_unit_cost,
    switches=(
        Switch(
            "temporary_members",
            label="Temporary workforce mix",
        ),
        Switch(
            "temporary_hours",
            label="Temporary regular hours",
        ),
        Switch(
            "temporary_rate",
            label="Temporary regular rates",
        ),
        Switch(
            "overtime_hours",
            label="Temporary overtime hours",
        ),
        Switch(
            "overtime_rate",
            label="Temporary overtime rates",
        ),
    ),
)

mix = value("employment_mix")

team_game = Game(
    name="team",
    formula=(
        select(mix, "employment_kind", "permanent")
        * child("permanent")
        + select(mix, "employment_kind", "gpd")
        * child("temp_game")
    ),
    switches=(
        Switch(
            "employment_mix",
            label="Permanent/temporary composition",
        ),
        Switch(
            "permanent_cost",
            child="permanent",
            label="Permanent unit cost",
        ),
        Switch(
            "temporary_cost",
            child="temp_game",
            label="Temporary unit cost",
        ),
    ),
)

model = ModelSpec(
    model_id="simple-payroll",
    version="1.0.0",
    graph=ModelGraph(
        source=source,
        populations=populations,
        fields=fields,
        row_costs=row_costs,
        factors=factors,
        games=(
            permanent_game,
            temporary_game,
            team_game,
        ),
        team_game="team",
        organization=organization_game("team"),
        local=local_game("team"),
        missing_economics="carry_observed",
    ),
)

In [5]:
compiled = compile_model(model)
description = compiled.inspect()
display(pd.DataFrame(description["games"])[
    ["id", "population", "switches"]
])

,id,population,switches
0,permanent,permanent,"[grade_mix, region_within_grade_mix, permanent..."
1,temp_game,gpd,"[temporary_members, temporary_hours, temporary..."
2,team,all,"[employment_mix, permanent_cost, temporary_cost]"


## 3. Supply the original mock rows

There is only one team in this dataset. We do not invent extra teams for a
more elaborate chart. Employee `p2` changes grade/cell; temporary employee
`t2` appears in September. Missing fields after concatenation belong to the
other population and are intentionally inapplicable.

The supplied `actual_cost` is an independent row-level reconciliation reference.
Expected totals are **6,575 → 8,374**, a **+1,799** change. The experiment's
commented-out +407 assertion belongs to an earlier example, not these rows.

In [6]:
permanent_data = pd.DataFrame(
    [
        ("2026-08-01", "p1", "G1", 0.8, 10., 100., 50., 1050.),
        ("2026-08-01", "p2", "G1", 0.8, 20., 100., 100., 2100.),
        ("2026-08-01", "p3", "G2", 1., 25., 80., 0., 2000.),
        ("2026-09-01", "p1", "G1", 0.8, 12., 110., 60., 1380.),
        ("2026-09-01", "p2", "G2", 1., 21., 90., 80., 1970.),
        ("2026-09-01", "p3", "G2", 1., 25., 80., 0., 2000.),
    ],
    columns=[
        "period",
        "employee_id",
        "grade",
        "reg_coeff",
        "stavka_oklad",
        "vyrabotka_act",
        "bonus_kpi",
        "actual_cost",
    ],
).assign(
    employment_type="permanent",
    team="Operations",
)

temporary_data = pd.DataFrame(
    [
        ("2026-08-01", "t1", 80., 15., 10., 22.5, 1425.),
        ("2026-09-01", "t1", 90., 16., 8., 24., 1632.),
        ("2026-09-01", "t2", 75., 16., 8., 24., 1392.),
    ],
    columns=[
        "period",
        "employee_id",
        "temporary_hours",
        "temporary_rate",
        "overtime_hours",
        "overtime_rate",
        "actual_cost",
    ],
).assign(
    employment_type="gpd",
    team="Operations",
)

frame = pd.concat(
    [permanent_data, temporary_data],
    ignore_index=True,
)

snapshot = DatasetSnapshot(
    dataset_id="simple-payroll-data",
    version="1",
    frame=frame,
)

In [7]:
display(frame)
source_totals = frame.groupby("period")["actual_cost"].sum()
display(source_totals.rename("Recorded payroll").to_frame())
np.testing.assert_allclose(source_totals, [6575.0, 8374.0])

,period,employee_id,grade,reg_coeff,stavka_oklad,vyrabotka_act,bonus_kpi,actual_cost,employment_type,team,temporary_hours,temporary_rate,overtime_hours,overtime_rate
0,2026-08-01,p1,G1,0.8,10.0,100.0,50.0,1050.0,permanent,Operations,NaN,NaN,NaN,NaN
1,2026-08-01,p2,G1,0.8,20.0,100.0,100.0,2100.0,permanent,Operations,NaN,NaN,NaN,NaN
2,2026-08-01,p3,G2,1.0,25.0,80.0,0.0,2000.0,permanent,Operations,NaN,NaN,NaN,NaN
3,2026-09-01,p1,G1,0.8,12.0,110.0,60.0,1380.0,permanent,Operations,NaN,NaN,NaN,NaN
4,2026-09-01,p2,G2,1.0,21.0,90.0,80.0,1970.0,permanent,Operations,NaN,NaN,NaN,NaN
5,2026-09-01,p3,G2,1.0,25.0,80.0,0.0,2000.0,permanent,Operations,NaN,NaN,NaN,NaN
6,2026-08-01,t1,NaN,NaN,NaN,NaN,NaN,1425.0,gpd,Operations,80.0,15.0,10.0,22.5
7,2026-09-01,t1,NaN,NaN,NaN,NaN,NaN,1632.0,gpd,Operations,90.0,16.0,8.0,24.0
8,2026-09-01,t2,NaN,NaN,NaN,NaN,NaN,1392.0,gpd,Operations,75.0,16.0,8.0,24.0


,Recorded payroll
period,
2026-08-01,6575.0
2026-09-01,8374.0


## 4. Analyze once

Compile validates the declaration; `analyze` constructs the period states and
runs the games. This example explicitly uses exact attribution. Team selection
is not passed to analysis: all source rows participate in employee alignment.
Reporting below can be rerun without rerunning this cell.

In [8]:
request = AnalysisRequest(
    period_a="2026-08",
    period_b="2026-09",
    source_cost_column="actual_cost",
    explainer="exact",
)
result = analyze(compiled, snapshot, request)
assert result.table("validation")["passed"].all()
np.testing.assert_allclose(
    result.table("driver_bridge")["impact"].sum(),
    1799.0,
    rtol=1e-10,
    atol=1e-6,
)

/home/rupilbo/driver_model/production/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. One call for the notebook report

This displays the full organizational waterfall, driver table, team summary,
recorded validation/assumption/estimator summaries, and a separate local
waterfall for Operations. Hover bars for driver IDs and precise amounts;
use the Plotly toolbar to zoom or reset the view.

Orange means an increase and blue a decrease, without a favorable/unfavorable
judgment. Existing zero effects stay visible. `currency_label` is only a label;
the example does not infer a currency or convert money.

In [9]:
charts = report(
    result,
    teams=selected_teams,
    currency_label=currency_label,
    renderer=renderer,
)

driver,label,impact
workforce_scale,Workforce scale,"1,659.28"
team_mix,Team mix,0.00
team::employment_mix,Permanent/temporary composition,-190.01
permanent::grade_mix,Permanent grade composition,573.75
permanent::region_within_grade_mix,Permanent region composition within grade,0.00
permanent::permanent_members,Workforce mix within cell,-623.53
permanent::permanent_rate,Permanent hourly rates,310.50
permanent::permanent_hours,Permanent worked hours,-52.31
permanent::permanent_bonus,Permanent bonuses,-5.91
temp_game::temporary_members,Temporary workforce mix,-98.96


team,period_A_cost,period_B_cost,cost_change,exposure_A,exposure_B
Operations,"6,575.00","8,374.00","1,799.00",4.00,5.00


recorded_status,checks
Passed,13


method
None recorded


method,games,n_permutations,evaluation_budget,model_evaluations,coalitions
exact,9,Unavailable,"64, 32, 8, 4","66, 34, 10, 6","64, 32, 8, 4"


driver,label,impact
workforce_scale,Workforce scale,"1,659.27"
team::employment_mix,Permanent/temporary composition,-190.01
permanent::grade_mix,Permanent grade composition,573.75
permanent::region_within_grade_mix,Permanent region composition within grade,0.00
permanent::permanent_members,Workforce mix within cell,-623.53
permanent::permanent_rate,Permanent hourly rates,310.50
permanent::permanent_hours,Permanent worked hours,-52.31
permanent::permanent_bonus,Permanent bonuses,-5.91
temp_game::temporary_members,Temporary workforce mix,-98.96
temp_game::temporary_hours,Temporary regular hours,169.41


## 6. Select a driver by ID

The table below lists organizational leaves with team allocations. Copy an ID
from the `driver` column, not its label or a hierarchy parent. Workforce scale
and team mix are organization-wide drivers and do not have this team breakdown.

The temporary-membership switch changes participating employee profiles and
weights, including the new `t2` profile. It does not require categorical
diversity. Permanent membership acts within grade/`reg_coeff` cells on the
whole paired rate × hours + bonus expression. The rate/hour/bonus switches
measure matched-input changes under the model's carry assumptions.
These are accounting allocations, not causal estimates.

In [10]:
available_drivers = (
    result.table("bridge")
    .loc[lambda rows: rows["team"].notna(), ["driver", "label"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
display(available_drivers)
selected_driver = "temp_game::temporary_members"
assert selected_driver in set(available_drivers["driver"])

,driver,label
0,team::employment_mix,Permanent/temporary composition
1,permanent::grade_mix,Permanent grade composition
2,permanent::region_within_grade_mix,Permanent region composition within grade
3,permanent::permanent_members,Workforce mix within cell
4,permanent::permanent_rate,Permanent hourly rates
5,permanent::permanent_hours,Permanent worked hours
6,permanent::permanent_bonus,Permanent bonuses
7,temp_game::temporary_members,Temporary workforce mix
8,temp_game::temporary_hours,Temporary regular hours
9,temp_game::temporary_rate,Temporary regular rates


In [11]:
driver_figure = plot_driver_by_team(
    result,
    selected_driver,
    teams=selected_teams,
    currency_label=currency_label,
)
driver_figure.show(renderer=renderer)

There is one bar because the source contains one team. Its selected subtotal
matches the organizational driver total here. With several teams, the same
function shows their signed organizational contributions, including offsets;
it never divides by the driver's net total. These bars are not local impacts.

To include that chart in the full report, supply `driver`. `show=False` returns
the figures and copied tables without displaying them. Call
`selected_report.show(renderer=renderer)` later to display the whole report.

In [12]:
selected_report = report(
    result,
    teams=selected_teams,
    driver=selected_driver,
    currency_label=currency_label,
    show=False,
)
print("Figures:", list(selected_report.figures))
display(selected_report.tables["driver_by_team"])

Figures: ['organization', 'local/Operations', 'driver_by_team']


,team,driver,impact
0,Operations,temp_game::temporary_members,-98.9625


## 7. Organizational and local waterfalls are separate views

Reusable figure functions return ordinary Plotly figures without displaying
them automatically. Organizational leaves explain the full organization;
local leaves explain Operations' own change. Never add these two allocations.

In this single-team example their amounts agree (apart from a zero team-mix
term and different scale IDs). With multiple teams, organizational allocations
need not equal each team's own change. `teams=None` or `teams=[]` omits local
waterfalls and leaves team tables unrestricted; it never filters inputs.

In [13]:
organization_figure = plot_organization_waterfall(
    result, currency_label=currency_label,
)
local_figure = plot_team_waterfall(
    result, "Operations", currency_label=currency_label,
)
organization_figure.show(renderer=renderer)
local_figure.show(renderer=renderer)

## 8. Inspect the tables and recorded evidence

The report owns copies of its tables. Display formatting below does not round
stored calculations. Estimator information comes from the actual coalition
audit. Passing recorded checks is not a claim that every analytical diagnostic
is implemented; the [architecture gaps](../docs/architecture-gaps.md) remain
open.

In [14]:
display(selected_report.tables["driver_bridge"][
    ["driver", "label", "impact"]
].style.format({"impact": "{:,.2f}"}))
display(selected_report.tables["team_summary"])
for name in (
    "validation_summary", "assumption_summary", "estimator_summary",
):
    print(name)
    display(selected_report.tables[name])

,driver,label,impact
0,workforce_scale,Workforce scale,"1,659.28"
1,team_mix,Team mix,0.00
2,team::employment_mix,Permanent/temporary composition,-190.01
3,permanent::grade_mix,Permanent grade composition,573.75
4,permanent::region_within_grade_mix,Permanent region composition within grade,0.00
5,permanent::permanent_members,Workforce mix within cell,-623.53
6,permanent::permanent_rate,Permanent hourly rates,310.50
7,permanent::permanent_hours,Permanent worked hours,-52.31
8,permanent::permanent_bonus,Permanent bonuses,-5.91
9,temp_game::temporary_members,Temporary workforce mix,-98.96


,team,period_A_cost,period_B_cost,cost_change,exposure_A,exposure_B
0,Operations,6575.0,8374.0,1799.0,4.0,5.0


validation_summary


,recorded_status,checks
0,Passed,13


assumption_summary


,method
0,None recorded


estimator_summary


,method,games,n_permutations,evaluation_budget,model_evaluations,coalitions
0,exact,9,Unavailable,"64, 32, 8, 4","66, 34, 10, 6","64, 32, 8, 4"


## 9. Customize an ordinary Plotly figure

Use `.update_layout` and `.update_traces` as usual. This modifies only this
figure's presentation. Values and the original result are unchanged.
The longer labels benefit from extra height and angled ticks.

In [15]:
organization_figure.update_layout(
    template="plotly_white",
    title_text="Operations payroll: August → September 2026",
    height=700,
    font={"size": 13},
    margin={"b": 240},
)
organization_figure.update_xaxes(tickangle=-45)
organization_figure.show(renderer=renderer)

## 10. Optional self-contained HTML export

Set `html_path` to a destination such as `"payroll-report.html"` and rerun
this cell. The HTML embeds Plotly.js and chart data, so a browser can open it
without CDN access. In a hosted notebook, download the file through its file
browser. An existing destination is overwritten by Plotly; the default below
writes nothing.

In [16]:
html_path = None  # For example: "payroll-report.html"
if html_path is not None:
    organization_figure.write_html(html_path, include_plotlyjs=True)
    print("Saved:", Path(html_path).resolve())

## 11. Report from a portable saved run

A bundle stores tables and metadata; loading it does not reconstruct states
or rerun attribution. This demonstration uses a temporary directory that is
automatically removed. For a retained run, pass a new permanent destination
to `export_run` instead.

The same report works on the loaded result, and its chart data match the
original report. Styling from section 9 belongs to that individual figure;
it does not change the saved analytical result.

In [17]:
with TemporaryDirectory(prefix="labor-report-showcase-") as directory:
    bundle_path = export_run(result, Path(directory) / "run")
    restored = load_run(bundle_path)

assert restored.states is None
restored_report = report(
    restored,
    teams=selected_teams,
    driver=selected_driver,
    currency_label=currency_label,
    show=False,
)
for name, figure in selected_report.figures.items():
    assert figure.to_json() == restored_report.figures[name].to_json()
print("Loaded run:", restored.run_id)
print("Figures match; no internal states are required.")
restored_report.figures["organization"].show(renderer=renderer)

Loaded run: 5cb2869d-facf-4fa7-b6a7-72d51b3e3ec9
Figures match; no internal states are required.


## Next steps

Change `selected_driver` using the available IDs, or style any figure in
`selected_report.figures`. Display an existing report again with `.show()`.
To change the model or source rows, rerun the analysis before reporting.
For the complete interface and renderer options see the
[reporting reference](../docs/reporting.md).